# Gemma 4 E4B Legal Merge + VLM Validation (A100)
## Status: April 10, 2026 — Track 3 finalized ✓

**Goal**: Merge the trained legal LoRA adapter into the regular Gemma 4 E4B multimodal base on an A100,
while preserving the original vision, audio, and multimodal projector weights.

**This notebook is Track 3 — the Colab A100 merge path.**
It produces the canonical merged HF checkpoint that feeds Tracks 1 and 2.

---

## 3-Track Inference Architecture

| Track | Hardware | Notebook | Serves |
|-------|----------|----------|--------|
| **Track 1** | RTX 3060 Ti (8 GB VRAM) | `Gemma4_Serving_Inference_Eval.ipynb` §5–§7 | TurboQuant `:8090` — turbo3 KV, 3-bit, 24 576 token context |
| **Track 2** | Intel 10th gen CPU (i5-10500, UHD 630) | `Gemma4_Serving_Inference_Eval.ipynb` §4 | LiteRT-LM `:8070` — E4B 3.65 GB, XNNPACK AVX2, MTP 4-head |
| **Track 3** | Colab A100 (80 GB HBM2e) | **This notebook** | Merges LoRA → safetensors → GGUF + .litertlm for Tracks 1 + 2 |

## L1–L4 KV Compression Reference (Intel 10th gen i5 · 12 MB L3 cache)

| Level | KV Format | Context tokens (CPU L3) | Context tokens (RTX 3060 Ti 8 GB) |
|-------|-----------|------------------------|-----------------------------------|
| **L1** | fp16 | 32 | 8 192 |
| **L2** | Q8\_0 | 64 | 16 384 |
| **L3** | turbo3 | 256 | 24 576 |
| **L4** | turbo4 | 384 | 32 768 |

Track 2 (LiteRT) targets **L2 (Q8_0 · 64 tokens)** — the native precision for XNNPACK on Intel Comet Lake.
Track 1 (TurboQuant) defaults to **L3 (turbo3 · 256 tokens)** on the 3060 Ti.

---

## What this notebook produces → what the others consume

```
This notebook (Track 3 — Colab A100)
  ↓ §6   merge_and_unload()  → gemma4-legal-vlm-merged/  (BF16 safetensors, multimodal)
  ↓ §9   ai-edge-torch       → gemma4-legal.litertlm      → Semaj90/gemma4-legal-litert-lm
  ↓ §10  llama.cpp convert   → gemma4-legal-vlm-Q4_K_M.gguf + gemma4-legal-vlm-mmproj-BF16.gguf

Track 2: download gemma4-legal.litertlm  → litert-lm --port 8070
Track 1: copy Q4_K_M.gguf               → llama-server -ctk turbo3 -ctv turbo3 --port 8090

Serving benchmark (both tracks): Gemma4_Serving_Inference_Eval.ipynb §7
```

---

## Architecture

```
google/gemma-4-E4B-it (regular multimodal base, BF16 on A100)
  ├── language_model              ← apply legal LoRA adapter here
  ├── vision_tower               ← KEEP ORIGINAL (frozen, never trained)
  ├── audio_tower                ← KEEP ORIGINAL (frozen, never trained)
  └── multi_modal_projector      ← KEEP ORIGINAL
```

## Key Rule

There is **no manual tensor re-attach step** if the full regular Gemma 4 base is loaded.
The correct workflow is:
1. Strip the adapter down to language-only LoRA tensors
2. Load the full regular Gemma 4 base
3. Apply only the language adapter
4. Merge and save the full multimodal model
5. Validate image understanding with the merged model
6. Branch from the merged HF checkpoint into TRT-LLM, GGUF (.litertlm for LiteRT), or direct serving

## Prerequisites
- **Runtime**: Colab A100 80 GB recommended for the cleanest BF16 merge path
- **Adapter**: `Semaj90/gemma4-e4b-legal-grpo` on HF Hub or a local uploaded stripped adapter
- **HF Token**: Set in Colab Secrets as `HF_TOKEN`

## Steps
1. Install core dependencies for Gemma 4 + PEFT merge
2. Log into Hugging Face
3. Download the adapter and strip it to language-only LoRA
4. Load the regular Gemma 4 E4B base in BF16 on A100
5. Apply the legal adapter and confirm no multimodal LoRA remains
6. Merge and save the full multimodal checkpoint
7. Reload merged checkpoint + run VLM smoke test
8. Stage a TRT-LLM/Triton export bundle from merged safetensors
9. Export LiteRT `.litertlm` → upload to `Semaj90/gemma4-legal-litert-lm` (feeds Track 2)
10. Export GGUF (BF16 + Q4\_K\_M + mmproj) for llama.cpp / Ollama / TurboQuant (feeds Track 1)
10b. Write Ollama Modelfile for one-command local deployment
11. Video frame analysis demo (optional — shows multimodal pattern)
12. Package all artifacts to Google Drive before session ends
13. Upload merged checkpoint + GGUF to Hugging Face Hub

## ⚡ Session Run Order — Read This First (April 10, 2026)

> **Merge is complete.** `gemma4-legal-vlm-merged/` is on Colab disk. The session is live.  
> Colab **silently wipes disk on disconnect** — follow this order without delay.

---

### ✅ DONE — §12: Google Drive Save
Checkpoint saved to `MyDrive/gemma4-legal-vlm-artifacts/`. **~16 GB protected.**

---

### ▶️ NEXT — §10: GGUF Export → Track 1 (RTX 3060 Ti)
Produces two files needed to serve via TurboQuant locally:

| File | Size | Purpose |
|------|------|---------|
| `gemma4-legal-vlm-Q4_K_M.gguf` | ~5–6 GB | Main language model |
| `gemma4-legal-vlm-mmproj-BF16.gguf` | ~400 MB | Vision bridge for image+text |

After downloading to your local machine, start Track 1:
```bash
# L3 — turbo3 KV (3-bit, default for RTX 3060 Ti 8 GB)
llama-server \
  -m gemma4-legal-vlm-Q4_K_M.gguf \
  --mmproj gemma4-legal-vlm-mmproj-BF16.gguf \
  -ctk turbo3 -ctv turbo3 \
  --port 8090 \
  --n-gpu-layers 35
```
Then run **`Gemma4_Serving_Inference_Eval.ipynb` §5–§7** to benchmark turbo3 vs Q8_0.

---

### Priority 3 — §9: LiteRT Export → Track 2 (Intel 10th gen CPU)
Produces `gemma4-legal.litertlm` (~3.65 GB) and uploads it to `Semaj90/gemma4-legal-litert-lm`.  
On the Intel 10th gen machine (i5-10500) after upload:
```bash
litert-lm \
  --model gemma4-legal.litertlm \
  --port 8070 \
  --threads 6
```
Then run **`Gemma4_Serving_Inference_Eval.ipynb` §4** to verify `is_litert_ready()` picks it up.

> **If ai-edge-torch fails** (Gemma 4 E4B recipe not yet available): use the pre-built
> `litert-community/gemma-4-E2B-it-litert-lm` as a stopgap for Track 2 benchmarking.

---

### Priority 4 — §8: TRT-LLM Bundle (lowest urgency — just writes text files)
No conversion happens here. The TRT engine build must run locally anyway.  
Do this last or skip if you need to close the session.

---

### Skip — §11 (video frame demo)
Optional demo, produces no output artifacts.

---

### After all exports: §13
Upload the merged safetensors checkpoint to `Semaj90/gemma4-e4b-legal-vlm-merged` so it survives beyond Google Drive.

## ✅ Upstream Fix — unsloth PR #4807 (April 2026)

**Status**: MERGED — `Gemma4ClippableLinear` PEFT checkpoint fix is now in unsloth main.

The `pip install unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git` in §1 automatically picks up this fix.

**What changed**: PEFT's checkpoint loader now correctly traverses the `.linear` submodule inside `Gemma4ClippableLinear` wrappers. This means:
- `merge_and_unload()` works cleanly with vision+audio towers intact
- The custom `Gemma4LoraLinear` class in §4 is kept as a safety fallback but may no longer be strictly required
- Target modules regex approach: `r".*.audio_tower.*.(q_proj|v_proj).linear"` (per BenjaminBossan)

**No notebook changes needed** — just run cells in order. The install cell pulls the fix from git main.

Ref: [unsloth/unsloth#4807](https://github.com/unslothai/unsloth/pull/4807) (rolandtannous)

# Step 0: Check torch version BEFORE installing anything
# torchao>=0.8 requires PyTorch 2.7+ (register_constant, torch.int1)
import torch as _t
_v = [int(x) for x in _t.__version__.split('+')[0].split('.')[:2]]
_need_pin = _v[0] < 2 or (_v[0] == 2 and _v[1] < 7)
print(f'PyTorch {_t.__version__} — {"NEED torchao pin" if _need_pin else "OK"}')
del _t, _v

# Install Unsloth (must include PR #4807 ClippableLinear fix)
!pip uninstall unsloth mergekit mergekit-moe -y 2>/dev/null || true
!pip install --upgrade --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install bitsandbytes accelerate peft transformers huggingface_hub pillow

# CRITICAL: Pin torchao AFTER unsloth install (overrides the too-new version it pulled)
# torchao>=0.8 uses torch.utils._pytree.register_constant + torch.int1 (PyTorch 2.7+ only)
if _need_pin:
    !pip install torchao==0.7.0 --quiet --no-deps
    print('Pinned torchao==0.7.0 for PyTorch <2.7 compatibility')

# Install llama.cpp for GGUF conversion (with mmproj support)
import os
if not os.path.exists('llama.cpp'):
    !git clone --depth 1 https://github.com/ggml-org/llama.cpp.git
    !cd llama.cpp && pip install -r requirements.txt 2>/dev/null || true
else:
    !cd llama.cpp && git pull
    print('llama.cpp already cloned')

# Verify
import torch
print(f'\nPyTorch: {torch.__version__}')
try:
    import torchao; print(f'torchao:  {torchao.__version__}')
except Exception as e:
    print(f'torchao:  IMPORT FAILED — {e}')
print(f'CUDA: {torch.cuda.is_available()} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A"})')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB' if torch.cuda.is_available() else '')

In [ ]:
# Core Colab dependencies only.
# Keep llama.cpp out of the first cell so GPU/RAM stay focused on Unsloth + merge work.
import sys
import subprocess

subprocess.run([sys.executable, '-m', 'pip', 'uninstall', 'unsloth', 'mergekit', 'mergekit-moe', '-y'], check=False)
subprocess.run(
    [
        sys.executable, '-m', 'pip', 'install', '--upgrade', '--no-cache-dir',
        'unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git'
    ],
    check=True
 )
subprocess.run(
    [
        sys.executable, '-m', 'pip', 'install',
        'bitsandbytes', 'accelerate', 'peft', 'transformers',
        'huggingface_hub', 'pillow', 'safetensors', 'requests'
    ],
    check=True
 )

import torch

# Pin torchao to avoid torch.int1 AttributeError (requires PyTorch 2.7+)
torch_major, torch_minor = [int(x) for x in torch.__version__.split('.')[:2]]
if torch_major < 2 or (torch_major == 2 and torch_minor < 7):
    print(f'PyTorch {torch.__version__} detected (<2.7) — pinning torchao==0.7.0')
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'torchao==0.7.0', '--quiet'], check=False)
else:
    print(f'PyTorch {torch.__version__} — torchao version OK')

print(f'\nPyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A"})')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB' if torch.cuda.is_available() else '')
print('\nllama.cpp setup is deferred until the GGUF export cell.')
print('If imports fail after package installation in a fresh Colab session, restart the runtime once and rerun from Cell 1.')

## §2 — Hugging Face Login

**Like showing your ID at the library door**: Gemma 4 is a gated model — Google requires you to accept its licence before downloading it. Your Hugging Face token proves you agreed.

How to set it up (one-time):
1. Go to https://huggingface.co/settings/tokens and copy your token
2. Accept the Gemma 4 licence at https://huggingface.co/google/gemma-4-E4B-it
3. In Colab, click the **🔑 Secrets** icon in the left sidebar → add a secret named `HF_TOKEN` → paste your token

This cell reads the secret automatically. You never need to paste your token into the code itself.

> **If you see "Set HF_TOKEN in Colab Secrets"**: the secret wasn't found — add it via the sidebar key icon and rerun this cell.

In [ ]:
from huggingface_hub import login
from google.colab import userdata

try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token)
    print('Logged in via Colab Secrets')
except Exception:
    print('Set HF_TOKEN in Colab Secrets (key icon in sidebar)')
    login()

## §3 — Download Adapter + Strip to Language-Only LoRA

**Like editing a resume to remove the wrong job history**: the trained legal adapter (`Semaj90/gemma4-e4b-legal-grpo`) was fine-tuned on the text part of Gemma 4. But the adapter file might accidentally contain hooks into the vision camera or audio parts of the model too. This cell cleans those out so only the legal language skills survive.

**Why this matters**: Gemma 4 E4B is a multimodal model. Its architecture has four parts:
- `language_model` ← the part we trained on legal text — **keep these LoRA weights**
- `vision_tower` ← the eyes — **must not be touched by the adapter**
- `audio_tower` ← the ears — **must not be touched by the adapter**
- `multi_modal_projector` ← the bridge between eyes/ears and language — **must not be touched**

If the adapter sneaks LoRA hooks into the vision/audio parts and we merge them in, the model's ability to analyze document images will break.

---

**Two paths — pick ONE:**

### Path A: Upload a pre-stripped adapter from your local machine
If you already ran the stripping step and have `gemma4-legal-text-only-adapter/adapter_model.safetensors` locally, use Colab's file upload sidebar and set `USE_LOCAL_ADAPTER = True`.

### Path B: Download from HF and strip automatically (default)
Downloads `Semaj90/gemma4-e4b-legal-grpo` from Hugging Face, removes any tensor whose name contains `vision_tower`, `audio_tower`, or `multi_modal_projector`, and saves the clean result to `gemma4-e4b-legal-text-only-adapter/`.

In [ ]:
import os, json, shutil
from safetensors.torch import load_file, save_file

# ============================================================
# TOGGLE: Set True if you uploaded the pre-stripped adapter
#         from c:/Users/james/Downloads/gemma4-legal-text-only-adapter/
# ============================================================
USE_LOCAL_ADAPTER = False

ADAPTER_DIR = 'gemma4-e4b-legal-grpo-lora'
TEXT_ONLY_DIR = 'gemma4-e4b-legal-text-only-adapter'
BLOCKED_MODULE_FRAGMENTS = ('vision_tower', 'audio_tower', 'multi_modal_projector', 'embed_vision', 'embed_audio')

def enforce_language_only_config(config_path):
    with open(config_path) as f:
        config = json.load(f)

    original_targets = config.get('target_modules')
    removed_targets = []
    if isinstance(original_targets, list):
        filtered_targets = [
            name for name in original_targets
            if not any(fragment in name for fragment in BLOCKED_MODULE_FRAGMENTS)
        ]
        removed_targets = [name for name in original_targets if name not in filtered_targets]
        if not filtered_targets:
            raise ValueError('Filtered target_modules is empty. Fix adapter_config.json before merge.')
        config['target_modules'] = filtered_targets
    else:
        print('target_modules is not a list; leaving it unchanged. Verify adapter_config.json manually before merge.')

    config['exclude_modules'] = list(BLOCKED_MODULE_FRAGMENTS)
    with open(config_path, 'w') as f:
        json.dump(config, f, indent=2)

    return removed_targets, config.get('target_modules')

if USE_LOCAL_ADAPTER:
    # ---- Path A: Use uploaded pre-stripped adapter ----
    assert os.path.exists(f'{TEXT_ONLY_DIR}/adapter_model.safetensors'), (
        f'Upload adapter_model.safetensors to {TEXT_ONLY_DIR}/ first!\n'
        'Local path: c:/Users/james/Downloads/gemma4-legal-text-only-adapter/'
    )
    assert os.path.exists(f'{TEXT_ONLY_DIR}/adapter_config.json'), (
        f'Upload adapter_config.json to {TEXT_ONLY_DIR}/ alongside the safetensors file!\n'
        'PEFT needs both files to load the adapter correctly.'
    )

    removed_targets, kept_targets = enforce_language_only_config(
        f'{TEXT_ONLY_DIR}/adapter_config.json'
    )
    tensors = load_file(f'{TEXT_ONLY_DIR}/adapter_model.safetensors')
    print('Path A: Using uploaded pre-stripped adapter')
    print(f'  {len(tensors)} tensors ({sum(v.nelement() * v.element_size() for v in tensors.values()) / 1024**2:.1f} MB)')
    vis = [k for k in tensors if 'vision_tower' in k]
    aud = [k for k in tensors if 'audio_tower' in k]
    proj = [k for k in tensors if any(f in k for f in ('multi_modal_projector', 'embed_vision', 'embed_audio'))]
    print(f'  vision_tower:          {len(vis)} (should be 0)')
    print(f'  audio_tower:           {len(aud)} (should be 0)')
    print(f'  projector/embedder:    {len(proj)} (should be 0)')  # Gemma4 uses embed_vision/embed_audio, not multi_modal_projector
    if removed_targets:
        print(f'  target_modules trimmed: removed {len(removed_targets)} multimodal entries')
    print(f'  target_modules kept:   {len(kept_targets) if isinstance(kept_targets, list) else kept_targets}')
    del tensors

else:
    # ---- Path B: Download from HF Hub + strip ----
    if not os.path.exists(f'{ADAPTER_DIR}/adapter_model.safetensors'):
        print('Downloading adapter from HF Hub...')
        from huggingface_hub import snapshot_download
        snapshot_download('Semaj90/gemma4-e4b-legal-grpo', local_dir=ADAPTER_DIR)
        print(f'Downloaded to {ADAPTER_DIR}/')
    else:
        print(f'Adapter already at {ADAPTER_DIR}/')

    tensors = load_file(f'{ADAPTER_DIR}/adapter_model.safetensors')
    keys = sorted(tensors.keys())
    lang = {k: v for k, v in tensors.items() if 'language_model' in k}
    vis = [k for k in keys if 'vision_tower' in k]
    aud = [k for k in keys if 'audio_tower' in k]
    proj = [k for k in keys if any(f in k for f in ('multi_modal_projector', 'embed_vision', 'embed_audio'))]

    print(f'\nOriginal adapter: {len(keys)} tensors')
    print(f'  language_model:        {len(lang)}')
    print(f'  vision_tower:          {len(vis)} (untrained — will strip LoRA only)')
    print(f'  audio_tower:           {len(aud)} (untrained — will strip LoRA only)')
    print(f'  projector/embedder:    {len(proj)} (untrained -- will strip LoRA only)')  # Gemma4 naming

    os.makedirs(TEXT_ONLY_DIR, exist_ok=True)
    save_file(lang, f'{TEXT_ONLY_DIR}/adapter_model.safetensors')

    with open(f'{ADAPTER_DIR}/adapter_config.json') as f:
        config = json.load(f)
    with open(f'{TEXT_ONLY_DIR}/adapter_config.json', 'w') as f:
        json.dump(config, f, indent=2)

    removed_targets, kept_targets = enforce_language_only_config(
        f'{TEXT_ONLY_DIR}/adapter_config.json'
    )

    for fname in os.listdir(ADAPTER_DIR):
        if fname not in ('adapter_model.safetensors', 'adapter_config.json'):
            src = os.path.join(ADAPTER_DIR, fname)
            if os.path.isfile(src):
                shutil.copy2(src, os.path.join(TEXT_ONLY_DIR, fname))

    orig_mb = sum(v.nelement() * v.element_size() for v in tensors.values()) / 1024**2
    new_mb = sum(v.nelement() * v.element_size() for v in lang.values()) / 1024**2
    print(f'\nAdapter surgery: {len(keys)} -> {len(lang)} tensors')
    print(f'  {orig_mb:.1f} MB -> {new_mb:.1f} MB (saved {orig_mb - new_mb:.1f} MB)')
    if removed_targets:
        print(f'  target_modules trimmed: removed {len(removed_targets)} multimodal entries')
    print(f'  target_modules kept:   {len(kept_targets) if isinstance(kept_targets, list) else kept_targets}')
    print(f'\nLanguage LoRA saved to: {TEXT_ONLY_DIR}/')
    print('Vision/audio/projector weights will come from the FULL BASE MODEL (original, untouched)')

    del tensors, lang

stripped_tensors = load_file(f'{TEXT_ONLY_DIR}/adapter_model.safetensors')
bad_keys = [
    key for key in stripped_tensors
    if any(fragment in key for fragment in BLOCKED_MODULE_FRAGMENTS)
]
assert not bad_keys, f'Stripped adapter still has multimodal tensors: {bad_keys[:5]}'
del stripped_tensors

print(f'\n=== Ready for merge ===')
print(f'Text-only adapter: {TEXT_ONLY_DIR}/adapter_model.safetensors')
print(f'Adapter config:    {TEXT_ONLY_DIR}/adapter_config.json')
print('Base model will supply the original frozen vision/audio/projector tensors during merge.')

## §4 — Load the Regular Gemma 4 Base + Apply Legal Adapter

**Like putting a legal-specialist "hat" on the full Gemma 4 brain**: this cell loads the complete `google/gemma-4-E4B-it` model in BF16 precision from Hugging Face — all four parts (language, vision, audio, projector) intact — and then drops the language-only legal adapter on top of it.

**Why BF16 on A100?**
A100 has 80 GB of HBM2e GPU memory. Loading in BF16 (16-bit brain floats) fits the entire ~40 GB model without any compression tricks and gives the cleanest possible starting point for a merge. Running a 4-bit quantized model and then dequantizing it just to merge would add unnecessary noise.

**What the cell confirms after loading:**
- how many LoRA parameters are attached
- that exactly **0** of them touch `vision_tower`, `audio_tower`, or `multi_modal_projector`
- if any multimodal LoRA hooks snuck through, it raises a hard error instead of silently corrupting the merge

**GPU note — Blackwell warning:** If Colab gave you a G4 (RTX PRO 6000 Blackwell, sm_120), you'll see a warning. The Unsloth kernels and bitsandbytes are not yet fully tested on sm_120 as of April 2026. Switch to an A100 Pro+ runtime if you hit CUDA errors here.

In [ ]:
import os
import torch
import warnings
import torch.nn as nn

from transformers import AutoModelForCausalLM, AutoProcessor
from peft import PeftModel, PeftConfig, LoraConfig
from peft.tuners.lora.layer import LoraLayer
from transformers.models.gemma4.modeling_gemma4 import Gemma4ClippableLinear

try:
    from safetensors.torch import load_file as safe_load_file
    HAS_SAFETENSORS = True
except Exception:
    HAS_SAFETENSORS = False

warnings.filterwarnings("ignore", category=UserWarning, message=".*torchao.*")

BASE_MODEL_ID = "google/gemma-4-E4B-it"
TEXT_ONLY_DIR = "gemma4-e4b-legal-text-only-adapter"
BLOCKED_MODULE_FRAGMENTS = ("vision_tower", "audio_tower", "multi_modal_projector", "embed_vision", "embed_audio")
DTYPE = torch.bfloat16

def _find_inner_linear(module: nn.Module) -> nn.Linear:
    if hasattr(module, "linear") and isinstance(module.linear, nn.Linear):
        return module.linear
    for child in module.modules():
        if child is module:
            continue
        if isinstance(child, nn.Linear):
            return child
    raise AttributeError(f"Could not find inner nn.Linear inside {type(module).__name__}")

class Gemma4LoraLinear(nn.Module, LoraLayer):
    def __init__(self, base_layer, adapter_name, r=0, lora_alpha=1, lora_dropout=0.0, **kwargs):
        nn.Module.__init__(self)
        LoraLayer.__init__(self, base_layer)

        self.base_layer = base_layer
        self.inner_linear = _find_inner_linear(base_layer)
        self.in_features = self.inner_linear.in_features
        self.out_features = self.inner_linear.out_features

        self.update_layer(
            adapter_name,
            r,
            lora_alpha,
            lora_dropout,
            kwargs.get("init_lora_weights", True),
            use_rslora=kwargs.get("use_rslora", False),
            use_dora=kwargs.get("use_dora", False),
            lora_bias=kwargs.get("lora_bias", False),
        )

    def forward(self, x: torch.Tensor, *args, **kwargs):
        result = self.base_layer(x, *args, **kwargs)
        for active_adapter in self.active_adapters:
            if active_adapter not in self.lora_A:
                continue
            lora_A = self.lora_A[active_adapter]
            lora_B = self.lora_B[active_adapter]
            dropout = self.lora_dropout[active_adapter]
            scaling = self.scaling[active_adapter]

            x_cast = x.to(lora_A.weight.dtype)
            delta = lora_B(lora_A(dropout(x_cast))) * scaling
            result = result + delta.to(result.dtype)
        return result

def extract_language_targets_from_adapter(adapter_dir: str):
    candidates = [
        os.path.join(adapter_dir, "adapter_model.safetensors"),
        os.path.join(adapter_dir, "adapter_model.bin"),
    ]

    state_dict = None
    for path in candidates:
        if os.path.exists(path):
            if path.endswith(".safetensors"):
                if not HAS_SAFETENSORS:
                    raise RuntimeError("adapter_model.safetensors found but safetensors is not installed")
                state_dict = safe_load_file(path)
            else:
                state_dict = torch.load(path, map_location="cpu")
            print(f"Loaded adapter state dict from: {path}")
            break

    if state_dict is None:
        raise FileNotFoundError("Could not find adapter_model.safetensors or adapter_model.bin")

    target_roots = set()
    for key in state_dict.keys():
        if not ("lora_A." in key or "lora_B." in key):
            continue
        if any(fragment in key for fragment in BLOCKED_MODULE_FRAGMENTS):
            continue
        root = key.split(".lora_A.")[0].split(".lora_B.")[0]
        prefixes_to_strip = ["base_model.model.", "base_model."]
        for prefix in prefixes_to_strip:
            if root.startswith(prefix):
                root = root[len(prefix):]
        target_roots.add(root)

    return sorted(target_roots)

assert torch.cuda.is_available(), "A CUDA GPU is required for this notebook."
print(f"GPU: {torch.cuda.get_device_name(0)}")

print(f"Loading regular Gemma 4 base: {BASE_MODEL_ID}")
processor = AutoProcessor.from_pretrained(BASE_MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=DTYPE,
    device_map="auto",
    trust_remote_code=True,
)
model.eval()

print(f"\nLoading PEFT config from: {TEXT_ONLY_DIR}")
peft_cfg = PeftConfig.from_pretrained(TEXT_ONLY_DIR)
lora_cfg = LoraConfig(**peft_cfg.to_dict())

if hasattr(lora_cfg, "_register_custom_module"):
    lora_cfg._register_custom_module({Gemma4ClippableLinear: Gemma4LoraLinear})
    print("Registered Gemma4LoraLinear fix")

exact_targets = extract_language_targets_from_adapter(TEXT_ONLY_DIR)
print(f"Exact language-only targets from adapter: {len(exact_targets)}")
if not exact_targets:
    raise RuntimeError("No language-only LoRA targets found in adapter checkpoint")

lora_cfg.target_modules = exact_targets

print(f"\nApplying language-only legal adapter from: {TEXT_ONLY_DIR}")
model = PeftModel.from_pretrained(
    model,
    TEXT_ONLY_DIR,
    config=lora_cfg,
    is_trainable=False,
)
model.eval()

lora_params = [n for n, p in model.named_parameters() if "lora_" in n]
print(f"LoRA parameters loaded: {len(lora_params)}")

bad_lora = [n for n in lora_params if any(f in n for f in BLOCKED_MODULE_FRAGMENTS)]
if bad_lora:
    raise RuntimeError(f"Still found multimodal LoRA hooks: {bad_lora[:10]}")

print("\nLanguage-only adapter successfully isolated.")
print("Ready to run multimodal validation and then merge.")

## 5. Quick Vision Inference Test (Pre-Merge)

Verify the model can process images before committing to the merge.

In [ ]:
from PIL import Image
import requests
from io import BytesIO

# Download a legal-document-style test image
test_url = 'https://upload.wikimedia.org/wikipedia/commons/thumb/3/3b/Constitution_of_the_United_States%2C_page_1.jpg/800px-Constitution_of_the_United_States%2C_page_1.jpg'
try:
    img_data = requests.get(test_url, timeout=10).content
    test_image = Image.open(BytesIO(img_data)).convert('RGB')
    print(f'Test image: {test_image.size} ({len(img_data) / 1024:.0f} KB)')
except Exception as e:
    print(f'Image download failed: {e}')
    test_image = Image.new('RGB', (768, 768), (200, 200, 200))
    print('Using placeholder image')

prompt = (
    'Analyze this legal document image. Identify the document type, '
    'summarize visible text or structure, and note any legal-significant details.'
)

messages = [
    {'role': 'user', 'content': [
        {'type': 'image', 'image': test_image},
        {'type': 'text', 'text': prompt},
    ]}
 ]

inputs = processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors='pt',
    enable_thinking=False,
 ).to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=1.0,
        top_p=0.95,
        top_k=64,
        do_sample=True,
    )

response = processor.decode(
    outputs[0][inputs['input_ids'].shape[1]:],
    skip_special_tokens=True
 )
print(f'\n=== VLM Response (adapter attached, pre-merge) ===\n{response}')

## 6. Merge Adapter Into Regular Gemma 4 and Save Merged HF Checkpoint

On A100, the clean merge path is:
1. Load `google/gemma-4-E4B-it` in BF16
2. Apply the stripped legal adapter
3. Run `merge_and_unload()`
4. Save a standard merged Hugging Face checkpoint
5. Re-open that merged checkpoint for VLM validation and downstream export

This notebook is now optimized for the regular Gemma 4 merge path.
LiteRT export and evaluation will be handled separately after this merged checkpoint is validated.

In [ ]:
import os
import gc
import json
import torch
from peft import PeftModel

CLEAN_DIR = 'gemma4-legal-vlm-merged'
os.makedirs(CLEAN_DIR, exist_ok=True)

assert torch.cuda.is_available(), 'A CUDA GPU is required for merge.'
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'Merge GPU: {gpu_name} ({vram_gb:.1f} GB VRAM)')

# Safely identify or recreate the PEFT wrapper
if 'peft_model' in globals() and hasattr(peft_model, 'merge_and_unload'):
    active_peft = peft_model
elif hasattr(model, 'merge_and_unload'):
    active_peft = model
else:
    print('Re-applying PEFT wrapper to model for merge...')
    active_peft = PeftModel.from_pretrained(
        model,
        TEXT_ONLY_DIR,
        config=lora_cfg,
        is_trainable=False,
    )
    active_peft.eval()

print(f'Active PEFT model type: {type(active_peft).__name__}')
print('Merging legal adapter into regular Gemma 4 base...')

merged_model = active_peft.merge_and_unload()
merged_model.eval()

# Verify multimodal parameters are intact after merge
total = sum(p.numel() for p in merged_model.parameters())
vision_params = sum(p.numel() for n, p in merged_model.named_parameters() if 'vision_tower' in n)
audio_params = sum(p.numel() for n, p in merged_model.named_parameters() if 'audio_tower' in n)
# Gemma 4 uses embed_vision/embed_audio, NOT multi_modal_projector
embedder_params = sum(p.numel() for n, p in merged_model.named_parameters() if 'embed_vision' in n or 'embed_audio' in n)
lang_params = sum(p.numel() for n, p in merged_model.named_parameters() if 'language_model' in n)

print(f'Merged model params: {total:,}')
print(f'  language_model:        {lang_params:,}')
print(f'  vision_tower:          {vision_params:,}')
print(f'  audio_tower:           {audio_params:,}')
print(f'  embed_vision+audio:    {embedder_params:,}  (Gemma4 multimodal embedder)')

if vision_params == 0:
    print('WARNING: Merged model has 0 vision_tower params.')
    print('         VLM inference will not work. Check base model loading.')

print(f'\nSaving merged HF checkpoint to {CLEAN_DIR}/ ...')
merged_model.save_pretrained(
    CLEAN_DIR,
    safe_serialization=True,
    max_shard_size='5GB',
)
processor.save_pretrained(CLEAN_DIR)

try:
    merged_model.generation_config.save_pretrained(CLEAN_DIR)
except Exception as e:
    print(f'generation_config save skipped: {e}')

manifest = {
    'base_model': BASE_MODEL_ID,
    'adapter_dir': TEXT_ONLY_DIR,
    'format': 'huggingface-safetensors',
    'dtype': 'bfloat16',
    'target_runtime': 'a100-bf16-merge',
    'modalities_preserved': {
        'vision_tower': vision_params > 0,
        'audio_tower': audio_params > 0,
        'embed_vision_audio': embedder_params > 0,  # Gemma4 uses embed_vision/embed_audio, not multi_modal_projector
    },
}
with open(os.path.join(CLEAN_DIR, 'merge_manifest.json'), 'w') as f:
    json.dump(manifest, f, indent=2)

saved_files = []
for root, _, files in os.walk(CLEAN_DIR):
    for name in files:
        path = os.path.join(root, name)
        saved_files.append((path, os.path.getsize(path)))

total_gb = sum(size for _, size in saved_files) / 1024**3
print(f'\nSaved {len(saved_files)} files ({total_gb:.1f} GB total) to {CLEAN_DIR}/')

# Update global references to the merged state
model = merged_model
peft_model = None

gc.collect()
torch.cuda.empty_cache()
print('Merged regular Gemma 4 checkpoint is ready for clean reload and VLM validation.')


## §7 — Reload Merged Checkpoint and Validate VLM

**Like pressing Save on a document and then reopening it to make sure nothing got scrambled**: this cell loads the `gemma4-legal-vlm-merged/` folder from disk as a fresh model — no adapter in memory, no LoRA layer — and runs the same Constitution image prompt again.

**Why this step matters:**
The merge writes safetensors files to disk. This cell proves that what was written is a valid, loadable multimodal checkpoint — not corrupted, not missing the processor config, and still capable of understanding images. If the vision tower survived the merge intact, the output here should match what you saw in §5.

**If this test passes**, the merged checkpoint is the verified canonical artifact. You can safely use it as the source for:
- TensorRT-LLM and Triton text-serving (§8)
- LiteRT packaging (§9)
- GGUF export for llama.cpp or Ollama (§10)

**If this test fails**, something went wrong during save/reload. Do not continue to downstream exports — go back and check disk space and the §6 output logs.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoProcessor

CLEAN_DIR = 'gemma4-legal-vlm-merged'
assert os.path.exists(CLEAN_DIR), f'Merged checkpoint missing at {CLEAN_DIR}/'

print(f'Reloading merged checkpoint from {CLEAN_DIR}/ ...')
processor = AutoProcessor.from_pretrained(CLEAN_DIR)
model = AutoModelForCausalLM.from_pretrained(
    CLEAN_DIR,
    torch_dtype=torch.bfloat16,
    device_map='auto',
 )
model.eval()

reload_messages = [
    {'role': 'user', 'content': [
        {'type': 'image', 'image': test_image},
        {'type': 'text', 'text': (
            'Analyze this legal document image. Identify the document type, '
            'summarize visible content, and explain why the text appears legally relevant.'
        )},
    ]}
 ]

reload_inputs = processor.apply_chat_template(
    reload_messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors='pt',
    enable_thinking=False,
 ).to(model.device)

with torch.no_grad():
    reload_outputs = model.generate(
        **reload_inputs,
        max_new_tokens=256,
        temperature=1.0,
        top_p=0.95,
        top_k=64,
        do_sample=True,
    )

reload_response = processor.decode(
    reload_outputs[0][reload_inputs['input_ids'].shape[1]:],
    skip_special_tokens=True
 )
print(f'\n=== VLM Response (merged checkpoint reload) ===\n{reload_response}')

## §8 — Prepare TRT-LLM / Triton Export Bundle

**Like writing the instructions that tell your home GPU server how to use the merged model**: this cell does NOT convert anything. It writes a staging folder (`gemma4-legal-vlm-trt-export/`) with a README, smoke prompt, tool-call prompt, and manifest JSON — a ready-made kit to hand to your local TRT-LLM build pipeline.

**TRT-LLM is for fast text serving** — it's the engine behind your local server at `http://localhost:8099`. The conversion from safetensors → TRT engine happens outside Colab on your local rig (needs TensorRT installed and your specific GPU).

**KAG/RAG/DAG context injection** (wired in `sveltekit-frontend`):
Before any LLM call reaches TRT-LLM or TurboQuant, the server-side pipeline in `src/lib/server/rag-pipeline.ts` pre-assembles retrieval context:
- **RAG** — FAISS/pgvector dense semantic search over `evidence_items` Qdrant collection
- **KAG** — entity-linked knowledge traversal (statute, citation, case law nodes)
- **DAG** — dependency-ordered multi-hop reasoning over `yorha_evidence_nodes`

The bundled `tool_call_prompt.txt` demonstrates the expected agentic output format for these retrievals.

**What gets written:**
| File | Purpose |
|---|---|
| `README.md` | Step-by-step `trtllm-build` commands tailored to this model |
| `smoke_prompt.txt` | Legal prompt to hit `POST /v1/completions` on port 8099 |
| `tool_call_prompt.txt` | Agentic prompt comparing tool-call format against LiteRT (:8070) and TurboQuant (:8090) |
| `bundle_manifest.json` | Machine-readable record of source checkpoint, runtime target, and endpoints |

**Key reminder:** TRT-LLM only serves **text**. VLM (image+text) validation must stay in the merged HF branch or GGUF branch unless you build a separate multimodal TRT pipeline end-to-end.

In [ ]:
import os
import json
import textwrap

CLEAN_DIR = 'gemma4-legal-vlm-merged'
TRT_EXPORT_DIR = 'gemma4-legal-vlm-trt-export'
os.makedirs(TRT_EXPORT_DIR, exist_ok=True)

assert os.path.exists(CLEAN_DIR), f'Merged checkpoint missing at {CLEAN_DIR}/'

readme = textwrap.dedent(f'''
    # Gemma 4 Legal TRT-LLM / Triton Bundle

    Source checkpoint: {CLEAN_DIR}/
    Runtime target: TensorRT-LLM text serving behind Triton or the standalone TRT endpoint on port 8099.


    ## 1. Convert merged HF checkpoint to TRT-LLM checkpoint

    python examples/gemma/convert_checkpoint.py \\
      --model_dir /path/to/{CLEAN_DIR} \\
      --output_dir /path/to/trt_checkpoints/gemma4_legal \\
      --dtype bfloat16 \\
      --tp_size 1 \\
      --pp_size 1

    ## 2. Build engine with paged KV cache

    trtllm-build \\
      --checkpoint_dir /path/to/trt_checkpoints/gemma4_legal \\
      --output_dir /path/to/trt_engines/gemma4_legal \\
      --gemm_plugin auto \\
      --gpt_attention_plugin auto \\
      --max_batch_size 4 \\
      --max_input_len 4096 \\
      --max_output_len 1024 \\
      --paged_kv_cache enable \\
      --context_fmha enable \\
      --remove_input_padding enable

    ## 3. Smoke test your existing local endpoint

    curl http://localhost:8099/health

    curl -X POST http://localhost:8099/v1/completions \\
      -H 'Content-Type: application/json' \\
      -d '{{
        "prompt": "Summarize the evidentiary value of a notarized affidavit in two short paragraphs.",
        "max_tokens": 256,
        "temperature": 0.2,
        "stream": false
      }}'

    ## 4. Compare tool-call-friendly prompting

    Prompt file: tool_call_prompt.txt
    Goal: compare TRT text output against LiteRT tool-call output and merged-HF VLM output.
''').strip() + '\n'

smoke_prompt = (
    'Summarize the evidentiary value of a notarized affidavit in two short paragraphs. '
    'Focus on authentication, hearsay limits, and chain-of-custody implications.'
)

tool_call_prompt = (
    'You may call tools if needed. Available tools: glossary_search(query), case_search(query). '
    'Return either a direct answer or a structured tool call for the best next legal research step on '
    'chain of custody defects in a criminal case.'
)

with open(os.path.join(TRT_EXPORT_DIR, 'README.md'), 'w', encoding='utf-8') as f:
    f.write(readme)

with open(os.path.join(TRT_EXPORT_DIR, 'smoke_prompt.txt'), 'w', encoding='utf-8') as f:
    f.write(smoke_prompt + '\n')

with open(os.path.join(TRT_EXPORT_DIR, 'tool_call_prompt.txt'), 'w', encoding='utf-8') as f:
    f.write(tool_call_prompt + '\n')

bundle_manifest = {
    'source_checkpoint': CLEAN_DIR,
    'source_format': 'huggingface-safetensors',
    'target_runtime': 'trt-llm-triton-text',
    'local_endpoint': 'http://localhost:8099',
    'health_path': '/health',
    'completion_path': '/v1/completions',
    'notes': [
        'Use merged HF checkpoint as the canonical source artifact.',
        'Use GGUF only for llama.cpp or Ollama branches.',
        'Validate VLM separately from TRT text-serving until multimodal TRT wiring is proven.'
    ]
}

with open(os.path.join(TRT_EXPORT_DIR, 'bundle_manifest.json'), 'w', encoding='utf-8') as f:
    json.dump(bundle_manifest, f, indent=2)

print(f'TRT export bundle written to {TRT_EXPORT_DIR}/')
print('Files: README.md, smoke_prompt.txt, tool_call_prompt.txt, bundle_manifest.json')
print('Use the merged HF checkpoint for TRT-LLM conversion, then compare against the local 8099 endpoint.')

## §9 — Convert Merged Checkpoint → LiteRT (.litertlm) — Track 2 Output

**Like shrinking a full-sized legal textbook down to fit in your pocket**: LiteRT is Google's on-device AI format — a single `.litertlm` file that runs on phones, tablets, and edge hardware without a GPU server. This cell converts the merged safetensors checkpoint into that format.

**This is the link between this notebook (Track 3) and the LiteRT serving path (Track 2).**

```
gemma4-legal-vlm-merged/            ← produced by §6 (this notebook, A100)
  ↓  ai-edge-torch export
gemma4-legal-vlm-litert/gemma4-legal.litertlm
  ↓  upload to HF Hub
Semaj90/gemma4-legal-litert-lm
  ↓  Track 2 — Intel 10th gen CPU (i5‑10500 + UHD 630 iGPU)
litert-lm --model gemma4-legal.litertlm --port 8070 --threads 6
```

**Track 2 target hardware — Intel 10th gen (Comet Lake):**
- CPU: i5-10500 · 6 cores · 12 MB L3 cache · XNNPACK AVX2 acceleration
- iGPU: UHD 630 · OpenCL via OpenVINO (detected in `Gemma4_Serving_Inference_Eval.ipynb §3`)
- L2 compression = **Q8_0** = 64 token KV in L3 cache
- MTP 4-head decoding → 1.8× throughput vs single-head LiteRT-LM

**Important caveats:**
- `.litertlm` is **text-only** — the vision tower does not fit in the on-device format. Keep the merged HF checkpoint for multimodal work.
- Requires `ai-edge-torch >= 0.3` for confirmed Gemma 4 support. The cell checks the version and warns if it's too old.
- If the export fails, the cell sets `EXPORT_OK = False` and prints guidance — it does **not** crash the notebook. You can still continue to §10 GGUF.
- A100 (40+ GB VRAM) is needed because the full BF16 model must fit in memory during conversion.

**After upload:** Set the LiteRT eval notebook:
```python
CUSTOM_HF_REPO = "Semaj90/gemma4-legal-litert-lm"
CUSTOM_HF_FILE = "gemma4-legal.litertlm"
```
And in `Gemma4_Serving_Inference_Eval.ipynb` §4, `is_litert_ready()` will detect `:8070` automatically.

> **GPU note:** G4 Blackwell works fine here since this step is pure PyTorch + ai-edge-torch — no Unsloth or bitsandbytes kernels involved. Optionally switch runtimes after §8 if you want to save A100 quota.

In [ ]:
import gc
import json
import os
import subprocess
import sys
import torch

CLEAN_DIR = 'gemma4-legal-vlm-merged'
LITERT_DIR = 'gemma4-legal-vlm-litert'
LITERT_FILE = 'gemma4-legal.litertlm'
LITERT_PATH = os.path.join(LITERT_DIR, LITERT_FILE)
HF_LITERT_REPO = 'Semaj90/gemma4-legal-litert-lm'

os.makedirs(LITERT_DIR, exist_ok=True)
assert os.path.exists(CLEAN_DIR), (
    f'Merged checkpoint not found at {CLEAN_DIR}/. Run merge cell (§6) first.'
)

# ── Install ────────────────────────────────────────────────────────────────────
print('Installing ai-edge-torch[generative] ...')
subprocess.check_call(
    [sys.executable, '-m', 'pip', 'install', 'ai-edge-torch[generative]', '-q', '--upgrade'],
    stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT
)
import importlib
import ai_edge_torch

version_str = getattr(ai_edge_torch, '__version__', 'unknown')
print(f'ai-edge-torch version: {version_str}')
try:
    from packaging.version import Version
    if Version(version_str) < Version('0.3.0'):
        print(f'WARNING: {version_str} may not support Gemma 4. Recommend >= 0.3.0.')
except Exception:
    pass

# ── Locate Gemma module (path varies by version) ───────────────────────────────
# 0.3.x: generative.examples.gemma.gemma3 (covers Gemma 3 + 4 family)
# future: may split to gemma4 submodule
gemma_mod = None
for mod_path in (
    'ai_edge_torch.generative.examples.gemma.gemma4',
    'ai_edge_torch.generative.examples.gemma.gemma3',
):
    try:
        gemma_mod = importlib.import_module(mod_path)
        print(f'Gemma module: {mod_path}')
        break
    except ImportError:
        continue

if gemma_mod is None:
    print('\n=== Gemma 4 E4B LiteRT conversion not yet available ===')
    print('ai-edge-torch does not include a Gemma 4 module in this version.')
    print('Options:')
    print('  A. Pre-built: litert-community/gemma-4-E2B-it-litert-lm (use in eval notebook)')
    print('  B. https://github.com/google-ai-edge/ai-edge-torch/releases')
    print('  C. Try again after: pip install ai-edge-torch[generative] --upgrade')
    print('Skipping. §10 GGUF export works independently.')
    sys.exit(0)

# ── Locate E4B config ──────────────────────────────────────────────────────────
model_config = None
for fn_name in ('get_model_config_4b', 'get_model_config_e4b', 'get_model_config_2b'):
    fn = getattr(gemma_mod, fn_name, None)
    if fn is not None:
        model_config = fn()
        print(f'Config: {fn_name}()')
        break

if model_config is None:
    print('ERROR: No E4B config found in Gemma module.')
    print('Available:', [a for a in dir(gemma_mod) if 'config' in a.lower()])
    sys.exit(0)

model_config.max_seq_len = 4096

# ── Build edge model ───────────────────────────────────────────────────────────
print(f'\nBuilding edge model from {CLEAN_DIR}/ ...')
build_fn = getattr(gemma_mod, 'build_model', None)
if build_fn is None:
    print('ERROR: build_model() not found in Gemma module.')
    sys.exit(0)
edge_model = build_fn(model_config, CLEAN_DIR)
edge_model.eval()
print('Edge model ready.')

# ── Export (two API paths — version dependent) ─────────────────────────────────
print(f'\nExporting to {LITERT_PATH} ...')
EXPORT_OK = False

try:
    # Path 1: utilities.export.export_to_litert (ai-edge-torch 0.3+)
    from ai_edge_torch.generative.utilities import export as litert_export
    export_fn = getattr(litert_export, 'export_to_litert', None)
    if export_fn is None:
        raise ImportError('export_to_litert not in utilities.export')
    export_fn(
        pytorch_model=edge_model,
        model_config=model_config,
        output_path=LITERT_PATH,
        prefill_seq_len=256,
        kv_cache_max_len=model_config.max_seq_len,
    )
    EXPORT_OK = True
    print('Path 1 (utilities.export_to_litert) succeeded.')
except Exception as e1:
    print(f'Path 1 failed: {e1}')
    try:
        # Path 2: ai_edge_torch.convert() — older/alternate API
        from ai_edge_torch import convert
        sample_kw = {
            'tokens': torch.zeros((1, 256), dtype=torch.int32),
            'input_pos': torch.arange(256, dtype=torch.int32).unsqueeze(0),
        }
        tflite_model = convert(edge_model.cpu(), (sample_kw,))
        tflite_model.export(LITERT_PATH)
        EXPORT_OK = True
        print('Path 2 (ai_edge_torch.convert) succeeded.')
    except Exception as e2:
        print(f'Path 2 failed: {e2}')

if not EXPORT_OK:
    print('\n=== LiteRT export unavailable for Gemma 4 E4B on this version ===')
    print('Possible causes:')
    print('  - Gemma 4 E4B recipe not yet merged in ai-edge-torch')
    print('  - Static-shape tracing incompatible with multimodal checkpoint')
    print('Options:')
    print('  A. Pre-built: litert-community/gemma-4-E2B-it-litert-lm (eval notebook)')
    print('  B. https://github.com/google-ai-edge/ai-edge-torch/releases')
    print('  C. Wait for ai-edge-torch >= 0.4 with native E4B recipe')
    print('\nProceeding to §10 GGUF is safe — it works independently.')
else:
    size_gb = os.path.getsize(LITERT_PATH) / 1024**3
    print(f'Export: {LITERT_PATH} ({size_gb:.2f} GB)')

    manifest = {
        'source_checkpoint': CLEAN_DIR,
        'litert_file': LITERT_FILE,
        'hf_target_repo': HF_LITERT_REPO,
        'ai_edge_torch_version': version_str,
        'model_config': {'params': '4b', 'max_seq_len': model_config.max_seq_len},
        'note': 'Text-only. Vision/audio towers not in .litertlm format.',
    }
    with open(os.path.join(LITERT_DIR, 'litert_manifest.json'), 'w') as mf:
        json.dump(manifest, mf, indent=2)
    print('Manifest written.')

    from huggingface_hub import HfApi
    api = HfApi()
    api.create_repo(HF_LITERT_REPO, repo_type='model', exist_ok=True)
    api.upload_file(
        path_or_fileobj=LITERT_PATH,
        path_in_repo=LITERT_FILE,
        repo_id=HF_LITERT_REPO,
        repo_type='model',
    )
    api.upload_file(
        path_or_fileobj=os.path.join(LITERT_DIR, 'litert_manifest.json'),
        path_in_repo='litert_manifest.json',
        repo_id=HF_LITERT_REPO,
        repo_type='model',
    )
    print(f'Uploaded to https://huggingface.co/{HF_LITERT_REPO}')
    print(f'\nIn the LiteRT eval notebook set:')
    print(f'  CUSTOM_HF_REPO = "{HF_LITERT_REPO}"')
    print(f'  CUSTOM_HF_FILE = "{LITERT_FILE}"')

gc.collect()
torch.cuda.empty_cache()


## §10 — Export Multimodal GGUF — Track 1 Output

**Like converting the merged model into a format that local GPU servers understand**: GGUF is the file format used by llama.cpp, Ollama, and LM Studio. The Q4\_K\_M variant serves Track 1 via TurboQuant's `llama-server` on your RTX 3060 Ti.

**Only run this after §7 (VLM reload test) has passed.** The GGUF export reads from `gemma4-legal-vlm-merged/`, so a corrupt merge produces a corrupt GGUF.

---

### Tensor Dissect Findings (April 10, 2026)

Previous "VLM" GGUF exports were **text-only despite the filename**:

| File | Tensors | Vision tower? | Verdict |
|------|---------|---------------|--------|
| `gemma4-e4b-legal.Q4_K_M.gguf` | 720 | No | Text-only |
| `gemma4-legal-vlm-q4_k_m.gguf` (April 10) | 720 | No | Text-only (name says VLM, tensors say no) |
| **Real VLM GGUF (this cell's output)** | **~1100+** | **Yes** | **Full multimodal** |

**Why 720 tensors = text-only**: The base Gemma 4 E4B language model has ~720 tensors. A full VLM GGUF should have ~1100+ tensors: 720 language + ~300 vision/SigLIP + ~80 audio + projector tensors. The old exports ran `convert_hf_to_gguf.py` on a checkpoint that was missing the vision tower weights.

**Adapter split recap**:
- HF `Semaj90/gemma4-e4b-legal-grpo`: 588 text-only LoRA tensors (140 MB) — vision/audio stripped
- Local Downloads adapter: 296 vision/audio LoRA tensors (22 MB) — the stripped half
- This notebook §3–§6 merges the text LoRA onto the full base, preserving original frozen vision tower

---

### Known llama.cpp Issues (April 2026)

**mmproj CUDA SIGABRT** ([ggml-org/llama.cpp#21402](https://github.com/ggml-org/llama.cpp/issues/21402)):
Loading Gemma 4 with `--mmproj` crashes with SIGABRT in `clip_model_loader::load_tensors` on CUDA. Root causes:
1. Early Unsloth mmproj files had incorrect `image_max_pixels` < `image_min_pixels`
2. CUDA kernel compatibility issues with vision encoder tensors

**Workarounds**:
- Use `--no-mmproj` for text-only inference (works reliably)
- Download updated mmproj from Unsloth (checksums changed April 4)
- Do NOT use `--image-min-tokens` flag (conflicts with Gemma 4 non-causal attention)

**Audio support**: Still actively evolving as of April 2026. Text + image are stable.

---

### Three steps this cell performs

1. **BF16 GGUF** — converts merged safetensors → BF16-precision GGUF (large, ~16 GB, highest quality)
2. **Q4\_K\_M GGUF** — quantizes the BF16 GGUF down to 4-bit (~5–6 GB, fits on a 12 GB gaming GPU)
3. **mmproj GGUF** — extracts the multimodal projector into a separate file for image+text inference

### Track 1 deployment (RTX 3060 Ti)

```bash
# Full VLM — turbo3 KV (if mmproj CUDA is fixed)
llama-server \
  -m gemma4-legal-vlm-Q4_K_M.gguf \
  --mmproj gemma4-legal-vlm-mmproj-BF16.gguf \
  -ctk turbo3 -ctv turbo3 \
  --port 8090 \
  --n-gpu-layers 35

# Text-only fallback (if mmproj crashes on CUDA)
llama-server \
  -m gemma4-legal-vlm-Q4_K_M.gguf \
  -ctk turbo3 -ctv turbo3 \
  --port 8090 \
  --n-gpu-layers 35
```

### Verification after export

```bash
# Count tensors in exported GGUF (should be ~1100+ for VLM)
python -c "
import struct
with open('gemma4-legal-vlm-gguf/gemma4-legal-vlm-Q4_K_M.gguf','rb') as f:
    magic = f.read(4)
    version = struct.unpack('<I', f.read(4))[0]
    n_tensors = struct.unpack('<Q', f.read(8))[0]
    print(f'Tensors: {n_tensors} (720=text-only, 1100+=VLM)')
"
```

### GGUF runtime uses two files

```
gemma4-legal-vlm-Q4_K_M.gguf         ← language + vision brain (~1100+ tensors)
gemma4-legal-vlm-mmproj-BF16.gguf    ← vision bridge (separate projector)
```

> **Build note:** This cell builds `llama-quantize` and `llama-mtmd-cli` locally. First run takes extra minutes to compile. Uses `-DGGML_CUDA=ON` on Colab for GPU-accelerated quantization.


In [ ]:
import os, subprocess, sys, struct

# Prepare llama.cpp only when GGUF export is needed
if not os.path.exists('llama.cpp'):
    print('Cloning llama.cpp for GGUF export...')
    subprocess.run(
        ['git', 'clone', '--depth', '1', 'https://github.com/ggml-org/llama.cpp.git'],
        check=True
    )
else:
    print('Refreshing llama.cpp checkout...')
    subprocess.run(['git', '-C', 'llama.cpp', 'pull'], check=False)

req_path = 'llama.cpp/requirements.txt'
if os.path.exists(req_path):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', req_path, '-q'], check=False)

CLEAN_DIR = 'gemma4-legal-vlm-merged'
GGUF_DIR = 'gemma4-legal-vlm-gguf'
os.makedirs(GGUF_DIR, exist_ok=True)

bf16_path = os.path.join(GGUF_DIR, 'gemma4-legal-vlm-BF16.gguf')
q4_path = os.path.join(GGUF_DIR, 'gemma4-legal-vlm-Q4_K_M.gguf')
mmproj_path = os.path.join(GGUF_DIR, 'gemma4-legal-vlm-mmproj-BF16.gguf')

# == Step 1: Convert safetensors -> BF16 GGUF (full multimodal model) =========
print('Step 1/3: Converting merged BF16 safetensors -> BF16 GGUF...')
result = subprocess.run(
    ['python', 'llama.cpp/convert_hf_to_gguf.py', CLEAN_DIR,
     '--outfile', bf16_path, '--outtype', 'bf16'],
    capture_output=True, text=True
)
if result.returncode == 0:
    fsize = os.path.getsize(bf16_path)
    print(f'  BF16 GGUF: {fsize / 1024**3:.1f} GB')
else:
    print(f'  ERROR: {result.stderr[:500]}')
    print('  Trying alternative conversion...')
    result = subprocess.run(
        ['python', 'llama.cpp/convert_hf_to_gguf.py', CLEAN_DIR,
         '--outfile', bf16_path, '--outtype', 'bf16', '--model-type', 'gemma4'],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        fsize = os.path.getsize(bf16_path)
        print(f'  BF16 GGUF (alt): {fsize / 1024**3:.1f} GB')
    else:
        print(f'  Fallback also failed: {result.stderr[:500]}')

# == Tensor count verification ================================================
if os.path.exists(bf16_path):
    with open(bf16_path, 'rb') as gf:
        magic = gf.read(4)
        version = struct.unpack('<I', gf.read(4))[0]
        n_tensors = struct.unpack('<Q', gf.read(8))[0]
    print(f'  Tensor count: {n_tensors}')
    if n_tensors < 800:
        print(f'  WARNING: Only {n_tensors} tensors -- likely TEXT-ONLY (no vision tower).')
        print(f'  A full VLM GGUF should have ~1100+ tensors.')
        print(f'  Check that gemma4-legal-vlm-merged/ contains vision_tower weights.')
        print(f'  Re-run merge cell (section 6) if the merged checkpoint is text-only.')
    else:
        print(f'  Tensor count OK: {n_tensors} tensors = full VLM model')

# == Step 2: Build llama-quantize (with CUDA if available) ====================
quantize_bin = 'llama.cpp/build/bin/llama-quantize'
if os.path.exists(bf16_path) and not os.path.exists(quantize_bin):
    print('\nBuilding llama.cpp (quantize + mtmd-cli + server)...')
    import shutil
    cuda_available = shutil.which('nvcc') is not None
    cmake_args = [
        'cmake', 'llama.cpp', '-B', 'llama.cpp/build',
        '-DBUILD_SHARED_LIBS=OFF',
        '-DCMAKE_BUILD_TYPE=Release',
    ]
    if cuda_available:
        cmake_args.append('-DGGML_CUDA=ON')
        print('  CUDA detected -- building with GPU support')
    else:
        print('  No CUDA -- building CPU-only')

    subprocess.run(cmake_args, capture_output=True)
    subprocess.run(
        ['cmake', '--build', 'llama.cpp/build', '--config', 'Release',
         '-j4', '--clean-first',
         '--target', 'llama-quantize', 'llama-mtmd-cli', 'llama-server'],
        capture_output=True
    )
    if os.path.exists(quantize_bin):
        print('  Build OK')
    else:
        print('  Build failed -- falling back to python quantize')

# == Step 2b: Quantize BF16 -> Q4_K_M =========================================
if os.path.exists(bf16_path):
    print('\nStep 2/3: Quantizing BF16 -> Q4_K_M...')
    if os.path.exists(quantize_bin):
        result = subprocess.run(
            [quantize_bin, bf16_path, q4_path, 'Q4_K_M'],
            capture_output=True, text=True
        )
        if result.returncode == 0 and os.path.exists(q4_path):
            fsize = os.path.getsize(q4_path)
            print(f'  Q4_K_M GGUF: {fsize / 1024**3:.1f} GB')
        else:
            print(f'  Quantization failed: {result.stderr[:500]}')
    else:
        print('  llama-quantize not built -- skipping quantization')
        print('  Download pre-quantized from ggml-org/gemma-4-E4B-it-GGUF instead')

# == Step 3: Extract multimodal projector -> mmproj GGUF =======================
print('\nStep 3/3: Extracting multimodal projector...')

# llama.cpp moved the mmproj script in recent versions
mmproj_scripts = [
    'llama.cpp/tools/mtmd/convert_image_encoder_to_gguf.py',     # current (April 2026)
    'llama.cpp/examples/llava/convert_image_encoder_to_gguf.py', # legacy path
]
mmproj_script = None
for script_path in mmproj_scripts:
    if os.path.exists(script_path):
        mmproj_script = script_path
        break

if mmproj_script:
    print(f'  Using: {mmproj_script}')
    result = subprocess.run(
        ['python', mmproj_script, '--model_dir', CLEAN_DIR,
         '--output_path', mmproj_path],
        capture_output=True, text=True
    )
    if result.returncode == 0 and os.path.exists(mmproj_path):
        fsize = os.path.getsize(mmproj_path)
        print(f'  mmproj GGUF: {fsize / 1024**2:.0f} MB')
    else:
        print(f'  mmproj extraction failed: {result.stderr[:500]}')
        print('  NOTE: Gemma 4 mmproj is a known pain point (April 2026).')
        print('  See: https://github.com/ggml-org/llama.cpp/issues/21402')
        print('  Workaround: use text-only mode (no --mmproj flag)')
else:
    print('  mmproj script not found at expected paths:')
    for p in mmproj_scripts:
        print(f'    {p}')
    print('  Try: git -C llama.cpp pull  (script may have moved again)')

# == Tensor count verification on Q4_K_M ======================================
if os.path.exists(q4_path):
    with open(q4_path, 'rb') as gf:
        magic = gf.read(4)
        version = struct.unpack('<I', gf.read(4))[0]
        n_tensors = struct.unpack('<Q', gf.read(8))[0]
    print(f'\n  Q4_K_M tensor count: {n_tensors}')
    if n_tensors < 800:
        print(f'  FAIL: {n_tensors} tensors = text-only. Vision tower missing.')
    else:
        print(f'  PASS: {n_tensors} tensors = full VLM')

# == Summary ==================================================================
print('\n=== GGUF Export Summary ===')
for label, f in [('BF16', bf16_path), ('Q4_K_M', q4_path), ('mmproj', mmproj_path)]:
    if os.path.exists(f):
        sz = os.path.getsize(f)
        unit = 'GB' if sz > 500_000_000 else 'MB'
        val = sz / 1024**3 if unit == 'GB' else sz / 1024**2
        print(f'  {label}: {os.path.basename(f)} ({val:.1f} {unit})')
    else:
        print(f'  {label}: NOT CREATED')

print('\n=== Deployment ===')
if os.path.exists(mmproj_path):
    print('VLM mode (image+text):')
    print(f'  llama-server -m {q4_path} --mmproj {mmproj_path} -ctk turbo3 -ctv turbo3 --port 8090 --n-gpu-layers 35')
    print()
    print('NOTE: If mmproj crashes on CUDA (SIGABRT in clip_model_loader::load_tensors),')
    print('      use text-only mode below. See https://github.com/ggml-org/llama.cpp/issues/21402')
print('Text-only mode (always works):')
print(f'  llama-server -m {q4_path} -ctk turbo3 -ctv turbo3 --port 8090 --n-gpu-layers 35')


## §10b — Create Ollama Modelfile

**Like writing a recipe card so Ollama knows how to serve the model**: Ollama uses a `Modelfile` — a small text file — to describe the model, its system prompt, and any serving parameters. This cell generates it from the GGUF files that §10 created.

After this cell, deploying on your local machine is one command:
```bash
ollama create gemma4-legal-vlm:latest -f gemma4-legal-vlm-gguf/Modelfile
ollama run gemma4-legal-vlm:latest
```

The Modelfile sets a legal-systems-analyst system prompt so the model behaves appropriately for legal Q&A out of the box.

> **mmproj note (April 2026):** If the mmproj file was not created by §10 (or crashes on CUDA), the Modelfile omits the `PROJECTOR` line and the model runs text-only. Image inference requires the mmproj CUDA fix from [llama.cpp#21402](https://github.com/ggml-org/llama.cpp/issues/21402).


In [ ]:
import os

GGUF_DIR = 'gemma4-legal-vlm-gguf'
q4_path = os.path.join(GGUF_DIR, 'gemma4-legal-vlm-Q4_K_M.gguf')
mmproj_path = os.path.join(GGUF_DIR, 'gemma4-legal-vlm-mmproj-BF16.gguf')

# Determine which GGUF to use
model_gguf = q4_path if os.path.exists(q4_path) else os.path.join(GGUF_DIR, 'gemma4-legal-vlm-BF16.gguf')

modelfile = f'''FROM {os.path.basename(model_gguf)}
'''

# Add mmproj if it exists as separate file
if os.path.exists(mmproj_path):
    modelfile += f'''PROJECTOR {os.path.basename(mmproj_path)}
'''

modelfile += '''PARAMETER temperature 0.3
PARAMETER num_predict 4096
PARAMETER num_ctx 32768
PARAMETER stop <end_of_turn>
PARAMETER stop <eos>

TEMPLATE """{{- range .Messages }}
{{- if eq .Role "system" }}<start_of_turn>system
{{ .Content }}<end_of_turn>
{{- else if eq .Role "user" }}<start_of_turn>user
{{ .Content }}<end_of_turn>
{{- else if eq .Role "assistant" }}<start_of_turn>model
{{ .Content }}<end_of_turn>
{{- end }}
{{- end }}<start_of_turn>model
"""

SYSTEM """You are a legal AI assistant specialized in evidence analysis, case law research,
and legal document interpretation. You have been fine-tuned on legal reasoning tasks
including statutory interpretation, case analysis, evidence evaluation, and legal writing.

When analyzing images of legal documents, evidence photos, or exhibits:
- Identify the document type and key elements
- Extract relevant text, dates, signatures, and markings
- Note any anomalies, redactions, or chain-of-custody indicators
- Provide structured analysis suitable for legal proceedings
"""
'''

modelfile_path = os.path.join(GGUF_DIR, 'Modelfile')
with open(modelfile_path, 'w') as f:
    f.write(modelfile)

print('=== Modelfile Created ===')
print(modelfile)
print(f'\nSaved to: {modelfile_path}')
print(f'\n=== Deployment Commands ===')
print(f'cd {GGUF_DIR}')
print(f'ollama create gemma4-legal-vlm:latest -f Modelfile')
print(f'ollama run gemma4-legal-vlm:latest')

## §11 — Video Frame Analysis Test

**Like showing the AI a flip-book and asking it to describe the movie**: Gemma 4 processes video by breaking it into individual image frames and analyzing them in a batch. This cell demonstrates the frame extraction → batch inference pattern.

**How Gemma 4 handles video:**
- Each frame is treated as an image with a configurable visual token budget (70–1120 tokens per frame)
- Low budget (70–140 tokens/frame) = more frames, faster inference, less detail per frame
- The model can handle roughly 60 seconds of video at 1 FPS natively

**This is a demo cell** — it shows the pattern you'd use to analyze surveillance footage, body-cam video, or depositions recorded on video. In production you'd extract frames with `ffmpeg` and send the batch to your local Ollama endpoint instead.

> **Skip this cell** if you only need text and document-image inference. It is independent of §12 (packaging).

In [ ]:
# Video frame analysis demo
# In production: extract frames with ffmpeg, send batch to Ollama

import subprocess, os, time
from PIL import Image

def extract_frames(video_path, fps=1, max_frames=30):
    """Extract frames from video at given FPS using ffmpeg."""
    frames_dir = 'video_frames'
    os.makedirs(frames_dir, exist_ok=True)

    cmd = [
        'ffmpeg', '-i', video_path,
        '-vf', f'fps={fps}',
        '-frames:v', str(max_frames),
        '-q:v', '2',
        os.path.join(frames_dir, 'frame_%04d.jpg'),
        '-y'
    ]
    subprocess.run(cmd, capture_output=True)

    frames = sorted([
        os.path.join(frames_dir, f)
        for f in os.listdir(frames_dir)
        if f.endswith('.jpg')
    ])
    return [Image.open(f).convert('RGB') for f in frames[:max_frames]]


def analyze_video_frames(frames, model, processor, prompt=None):
    """Analyze video frames as a batch of images."""
    if prompt is None:
        prompt = (
            'These are sequential frames from a video recording. '
            'Describe what is happening in this footage. '
            'Note any persons, actions, objects, or events relevant to a legal investigation.'
        )

    content = [{'type': 'image', 'image': frame} for frame in frames]
    content.append({'type': 'text', 'text': prompt})

    messages = [{'role': 'user', 'content': content}]

    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors='pt',
    ).to(model.device)

    t0 = time.time()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.3,
            do_sample=True,
        )
    elapsed = time.time() - t0

    response = processor.decode(
        outputs[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True
    )
    return response, elapsed


print('=== Video Frame Analysis Demo ===')
print('Creating synthetic test frames...')

test_frames = []
colors = [(200, 100, 100), (100, 200, 100), (100, 100, 200), (200, 200, 100), (200, 100, 200)]
for color in colors:
    img = Image.new('RGB', (384, 384), color)
    test_frames.append(img)

print(f'Frames: {len(test_frames)} ({test_frames[0].size})')
print('\nNote: In production, use extract_frames() with real video files.')
print('Example: frames = extract_frames("evidence_recording.mp4", fps=1, max_frames=30)')
print('         response, elapsed = analyze_video_frames(frames, model, processor)')
print('\nFor Ollama deployment, send frames as base64 images array:')
print('  curl http://localhost:11434/api/generate -d \'{"model":"gemma4-legal-vlm","images":["<b64>","<b64>",...]}\'')
print('\nRecommended settings for video:')
print('  - Token budget: 70-140 per frame (speed over detail)')
print('  - FPS: 1-2 for surveillance, 0.5 for documents/static')
print('  - Max frames: 30 (Gemma4 handles up to ~60 at low budget)')

## §12 — Package All Artifacts for Download

**Like packing a suitcase before leaving the hotel**: Colab's disk is temporary — when the session ends, everything is gone. This cell lists every output file, shows its size, and optionally copies all artifacts to your Google Drive so nothing is lost.

**What gets collected:**
| Folder | Contents |
|---|---|
| `gemma4-legal-vlm-merged/` | The canonical merged HF safetensors checkpoint |
| `gemma4-legal-vlm-trt-export/` | TRT-LLM staging README + prompt files |
| `gemma4-legal-vlm-gguf/` | BF16 GGUF, Q4\_K\_M GGUF, mmproj GGUF, Modelfile |

**Google Drive mount:** the cell tries to mount your Drive and copy everything to `MyDrive/gemma4-legal-vlm-artifacts/`. If Drive is unavailable, it prints a manual fallback (zip the folders from the Colab sidebar).

> **Run this before closing the Colab tab.** The merged checkpoint is ~16 GB — too big to wait until the session warns you it's about to disconnect.

In [ ]:
import os
import shutil

GGUF_DIR = 'gemma4-legal-vlm-gguf'
CLEAN_DIR = 'gemma4-legal-vlm-merged'
TRT_EXPORT_DIR = 'gemma4-legal-vlm-trt-export'
ARTIFACT_DIRS = [CLEAN_DIR, TRT_EXPORT_DIR, GGUF_DIR]

print('=== Output Files ===')
for artifact_dir in ARTIFACT_DIRS:
    if not os.path.exists(artifact_dir):
        print(f'  {artifact_dir}/: NOT PRESENT')
        continue

    for root, _, files in os.walk(artifact_dir):
        rel_root = os.path.relpath(root, '.')
        for name in sorted(files):
            fpath = os.path.join(root, name)
            fsize = os.path.getsize(fpath)
            if fsize > 1024**3:
                size_str = f'{fsize / 1024**3:.2f} GB'
            else:
                size_str = f'{fsize / 1024**2:.1f} MB'
            print(f'  {os.path.join(rel_root, name)}: {size_str}')

print('\n=== Save to Google Drive ===')
try:
    from google.colab import drive
    drive.mount('/content/drive')

    DRIVE_DIR = '/content/drive/MyDrive/gemma4-legal-vlm-artifacts'
    os.makedirs(DRIVE_DIR, exist_ok=True)

    for artifact_dir in ARTIFACT_DIRS:
        if not os.path.exists(artifact_dir):
            continue
        dst_dir = os.path.join(DRIVE_DIR, artifact_dir)
        if os.path.exists(dst_dir):
            shutil.rmtree(dst_dir)
        shutil.copytree(artifact_dir, dst_dir)
        print(f'  Copied {artifact_dir}/ -> {dst_dir}')

    print(f'\nSaved to Google Drive: {DRIVE_DIR}')
except Exception as e:
    print(f'Google Drive not available: {e}')
    print('Use the Colab file browser or zip specific artifact folders manually.')

print('\n=== Local Deployment Branches ===')
print(f'1. Canonical merged HF checkpoint: {CLEAN_DIR}/')
print(f'2. TRT-LLM/Triton staging bundle: {TRT_EXPORT_DIR}/')
if os.path.exists(GGUF_DIR):
    print(f'3. Optional GGUF branch: {GGUF_DIR}/')
    print('   Ollama: ollama create gemma4-legal-vlm:latest -f gemma4-legal-vlm-gguf/Modelfile')
else:
    print('3. Optional GGUF branch: not created yet')

print('\nLiteRT comparison should use the separate LiteRT eval notebook and a .litertlm model artifact.')

## §13 — Upload Artifacts to Hugging Face Hub (Optional)

**Like publishing your work to a shared cloud library**: once the merged checkpoint is validated, this cell uploads it to your Hugging Face account so you can pull it from anywhere — including inside the LiteRT eval notebook, your local machine, or a Triton server.

**Upload order (follow this):**
1. **Merged HF checkpoint** → `Semaj90/gemma4-e4b-legal-vlm` (the canonical multimodal artifact)
2. **GGUF files** (optional) → `Semaj90/gemma4-e4b-legal-vlm-gguf` (only if you need llama.cpp / Ollama distribution)

**Do NOT upload the adapter folder** (`gemma4-e4b-legal-text-only-adapter/`) to the same repo — it's an intermediate artifact and would confuse anyone trying to use the merged checkpoint directly.

After uploading, the LiteRT eval notebook's `CUSTOM_HF_REPO` field should be set to the `.litertlm` repo created in §9, not this safetensors repo.

In [ ]:
from huggingface_hub import HfApi

api = HfApi()
CLEAN_DIR = 'gemma4-legal-vlm-merged'
GGUF_DIR = 'gemma4-legal-vlm-gguf'
MERGED_REPO_ID = 'Semaj90/gemma4-e4b-legal-vlm-merged'
GGUF_REPO_ID = 'Semaj90/gemma4-e4b-legal-vlm-GGUF'

if os.path.exists(CLEAN_DIR):
    print(f'Uploading merged HF checkpoint to {MERGED_REPO_ID}...')
    try:
        api.create_repo(repo_id=MERGED_REPO_ID, repo_type='model', exist_ok=True)
        api.upload_folder(
            folder_path=CLEAN_DIR,
            repo_id=MERGED_REPO_ID,
            repo_type='model',
            commit_message='Gemma 4 E4B legal merged safetensors checkpoint',
        )
        print(f'Uploaded merged checkpoint: https://huggingface.co/{MERGED_REPO_ID}')
    except Exception as e:
        print(f'Merged checkpoint upload failed: {e}')
else:
    print(f'Skipping merged upload: {CLEAN_DIR}/ not found')

if os.path.exists(GGUF_DIR):
    print(f'\nUploading optional GGUF artifacts to {GGUF_REPO_ID}...')
    try:
        api.create_repo(repo_id=GGUF_REPO_ID, repo_type='model', exist_ok=True)
        api.upload_folder(
            folder_path=GGUF_DIR,
            repo_id=GGUF_REPO_ID,
            repo_type='model',
            commit_message='Gemma 4 E4B legal VLM GGUF artifacts',
        )
        print(f'Uploaded GGUF: https://huggingface.co/{GGUF_REPO_ID}')
    except Exception as e:
        print(f'GGUF upload failed: {e}')
else:
    print(f'\nSkipping GGUF upload: {GGUF_DIR}/ not found')

## §14 — Local Deployment Bundle

Writes `app.py`, `requirements.txt`, `client_example.py`, and `README.md` to a `gemma4-legal-deploy/` folder, then zips it for download.

**Run this after §13 finishes.** The zip is ~5 KB — download it and place the model folder alongside it on your local machine.


In [ ]:
import os, zipfile, textwrap
from google.colab import files

BUNDLE_DIR = "gemma4-legal-deploy"
os.makedirs(BUNDLE_DIR, exist_ok=True)

# ── app.py ──────────────────────────────────────────────────────────────────
APP_PY = textwrap.dedent('''
    """
    gemma4-legal FastAPI server
    Endpoints:
      POST /chat               — text-only inference
      POST /analyze-document   — image + text (VLM)
    Requires: gemma4-legal-vlm-merged/ in the same directory.
    """
    import os, base64, io
    from contextlib import asynccontextmanager
    from fastapi import FastAPI, HTTPException
    from fastapi.middleware.cors import CORSMiddleware
    from pydantic import BaseModel
    from typing import Optional
    import torch
    from transformers import AutoProcessor, Gemma3ForConditionalGeneration, BitsAndBytesConfig
    from PIL import Image

    MODEL_DIR = os.environ.get("MODEL_DIR", "gemma4-legal-vlm-merged")
    DEVICE    = "cuda" if torch.cuda.is_available() else "cpu"
    MAX_NEW_TOKENS = int(os.environ.get("MAX_NEW_TOKENS", "512"))

    model_state = {}

    @asynccontextmanager
    async def lifespan(app: FastAPI):
        print(f"Loading model from {MODEL_DIR} on {DEVICE}...")
        bnb_cfg = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16,
        )
        model_state["processor"] = AutoProcessor.from_pretrained(MODEL_DIR)
        model_state["model"] = Gemma3ForConditionalGeneration.from_pretrained(
            MODEL_DIR,
            quantization_config=bnb_cfg,
            device_map="auto",
            torch_dtype=torch.bfloat16,
        )
        model_state["model"].eval()
        print("Model ready.")
        yield
        model_state.clear()

    app = FastAPI(title="gemma4-legal", lifespan=lifespan)
    app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

    class ChatRequest(BaseModel):
        prompt: str
        system: Optional[str] = "You are a legal AI assistant. Answer accurately and cite relevant legal principles."
        max_new_tokens: Optional[int] = MAX_NEW_TOKENS

    class AnalyzeRequest(BaseModel):
        prompt: str
        image_base64: str          # base64-encoded image (JPEG or PNG)
        system: Optional[str] = "You are a legal document analyst. Describe what you observe."
        max_new_tokens: Optional[int] = MAX_NEW_TOKENS

    def _generate(messages, max_new_tokens):
        processor = model_state["processor"]
        model     = model_state["model"]
        inputs = processor.apply_chat_template(
            messages, add_generation_prompt=True, tokenize=True,
            return_dict=True, return_tensors="pt"
        ).to(DEVICE)
        input_len = inputs["input_ids"].shape[-1]
        with torch.inference_mode():
            out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
        return processor.decode(out[0][input_len:], skip_special_tokens=True)

    @app.post("/chat")
    def chat(req: ChatRequest):
        try:
            messages = [
                {"role": "system",    "content": [{"type": "text", "text": req.system}]},
                {"role": "user",      "content": [{"type": "text", "text": req.prompt}]},
            ]
            return {"response": _generate(messages, req.max_new_tokens)}
        except Exception as e:
            raise HTTPException(status_code=500, detail=str(e))

    @app.post("/analyze-document")
    def analyze_document(req: AnalyzeRequest):
        try:
            img_bytes = base64.b64decode(req.image_base64)
            image = Image.open(io.BytesIO(img_bytes)).convert("RGB")
            # Conservative resize for 8 GB VRAM
            image.thumbnail((896, 896))
            messages = [
                {"role": "system", "content": [{"type": "text", "text": req.system}]},
                {"role": "user",   "content": [
                    {"type": "image", "image": image},
                    {"type": "text",  "text": req.prompt},
                ]},
            ]
            return {"response": _generate(messages, req.max_new_tokens)}
        except Exception as e:
            raise HTTPException(status_code=500, detail=str(e))

    @app.get("/health")
    def health():
        return {"status": "ok", "device": DEVICE, "model": MODEL_DIR}
''').strip()

with open(f"{BUNDLE_DIR}/app.py", "w") as f:
    f.write(APP_PY)
print("Written: app.py")

# ── requirements.txt ────────────────────────────────────────────────────────
REQS = textwrap.dedent("""
    fastapi>=0.111.0
    uvicorn[standard]>=0.29.0
    transformers>=4.47.0
    bitsandbytes>=0.43.0
    accelerate>=0.30.0
    pillow>=10.0.0
    python-multipart>=0.0.9
    torch>=2.2.0
    pydantic>=2.0.0
""").strip()

with open(f"{BUNDLE_DIR}/requirements.txt", "w") as f:
    f.write(REQS)
print("Written: requirements.txt")

# ── client_example.py ───────────────────────────────────────────────────────
CLIENT_PY = textwrap.dedent('''
    """
    Sample client for gemma4-legal FastAPI server.
    Usage:
      python client_example.py --text "What is chain of custody?"
      python client_example.py --image path/to/doc.jpg --prompt "Summarize this document."
    """
    import argparse, base64, json, sys
    import urllib.request

    BASE = "http://localhost:8000"

    def post(endpoint, payload):
        body = json.dumps(payload).encode()
        req  = urllib.request.Request(f"{BASE}{endpoint}", data=body,
                                       headers={"Content-Type": "application/json"}, method="POST")
        with urllib.request.urlopen(req, timeout=120) as r:
            return json.loads(r.read())

    def main():
        p = argparse.ArgumentParser()
        p.add_argument("--text",   help="Text prompt for /chat")
        p.add_argument("--image",  help="Path to image file for /analyze-document")
        p.add_argument("--prompt", default="Describe the legal significance of this document.")
        args = p.parse_args()

        if args.image:
            with open(args.image, "rb") as f:
                b64 = base64.b64encode(f.read()).decode()
            result = post("/analyze-document", {"prompt": args.prompt, "image_base64": b64})
        elif args.text:
            result = post("/chat", {"prompt": args.text})
        else:
            print("Provide --text or --image"); sys.exit(1)

        print("\\n=== Response ===")
        print(result["response"])

    if __name__ == "__main__":
        main()
''').strip()

with open(f"{BUNDLE_DIR}/client_example.py", "w") as f:
    f.write(CLIENT_PY)
print("Written: client_example.py")

# ── README.md ────────────────────────────────────────────────────────────────
README = textwrap.dedent("""
    # gemma4-legal — Local Deployment

    FastAPI server wrapping `gemma4-legal-vlm-merged/` with 4-bit NF4 quantization.
    Supports text-only and image+text (VLM) inference on an 8 GB GPU.

    ## Setup

    ```bash
    # 1. Place files
    gemma4-legal-deploy/
    ├── app.py
    ├── requirements.txt
    ├── client_example.py
    └── README.md
    gemma4-legal-vlm-merged/   ← model folder (same parent directory)

    # 2. Install
    pip install -r requirements.txt

    # 3. Run
    uvicorn app:app --host 0.0.0.0 --port 8000
    ```

    ## Endpoints

    | Method | Path | Purpose |
    |--------|------|---------|
    | GET | `/health` | Check server + device status |
    | POST | `/chat` | Text-only legal Q&A |
    | POST | `/analyze-document` | Image + text document analysis |

    ## Sample calls

    ```bash
    # Text
    python client_example.py --text "What is chain of custody?"

    # Image
    python client_example.py --image evidence_photo.jpg --prompt "Describe what you see."
    ```

    ## Settings for RTX 3060 Ti (8 GB VRAM)

    - One request at a time (no batching)
    - Images auto-resized to max 896×896
    - Keep `max_new_tokens` ≤ 512
    - 4-bit NF4 only (already set in app.py)
    - If VRAM is still tight, reduce to `max_new_tokens=256` or switch to Gemma 4 E2B

    ## Text-only path (GGUF)

    For lighter text-only inference run llama-server alongside this server:

    ```bash
    llama-server \\
      -m gemma4-legal-vlm-q4_k_m.gguf \\
      -ctk turbo3 -ctv turbo3 \\
      --port 8090 \\
      --n-gpu-layers 35
    ```

    Route text-only requests to `:8090` and VLM requests to `:8000`.
""").strip()

with open(f"{BUNDLE_DIR}/README.md", "w") as f:
    f.write(README)
print("Written: README.md")

# ── Zip + download ───────────────────────────────────────────────────────────
ZIP_PATH = "gemma4-legal-deploy.zip"
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for fname in ["app.py", "requirements.txt", "client_example.py", "README.md"]:
        zf.write(f"{BUNDLE_DIR}/{fname}", fname)

print(f"\nBundle: {ZIP_PATH} ({os.path.getsize(ZIP_PATH)/1024:.1f} KB)")
files.download(ZIP_PATH)
